In [13]:
import pandas as pd
import matplotlib.pyplot as plt

In [14]:
# Read in the cleaned data
numeric_dtypes = {
    "Best of": "Int64",
    "Rank_1": "Int64",
    "Rank_2": "Int64",
    "Pts_1": "Int64",
    "Pts_2": "Int64",
    "Odd_1": "Float64",
    "Odd_2": "Float64",
    "Margin": "Float64",
    "Prob_1": "Float64",
    "Prob_2": "Float64",
    "WinRate180_1": "Float64",
    "WinRateLast5_365_1": "Float64",
    "WinRate180_2": "Float64",
}

wta = pd.read_csv(
    "../data/clean_wta.csv",
    dtype=numeric_dtypes,
    parse_dates=["Date"],
    na_values=["", "NA", "N/A", "null", "None", "-"],
    keep_default_na=True,
)


In [19]:
# Create Chronological train (80%) /validation (10%) /test splits (10%).

# Choose date boundaries near 80% and 90% of the wta rows.
validation_start = wta["Date"].iloc[int(len(wta) * 0.8)]
test_start = wta["Date"].iloc[int(len(wta) * 0.9)]

train = wta.loc[wta["Date"] < validation_start].copy()

validation = wta.loc[(wta["Date"] >= validation_start) & (wta["Date"] < test_start)].copy()

test = wta.loc[wta["Date"] >= test_start].copy()

# Check that all wta rows are assigned and the date ranges do not overlap.

assert len(train) + len(validation) + len(test) == len(wta)

assert train["Date"].max() < validation["Date"].min()

assert validation["Date"].max() < test["Date"].min()

split_summary = pd.DataFrame([
    {
        "Split": name,
        "Rows": len(part),
        "Percent": round(100 * len(part) / len(wta), 2),
        "Start": part["Date"].min(),
        "End": part["Date"].max(),
    }
    for name, part in [
        ("Train", train), ("Validation", validation), ("Test", test)
    ]
])

display(split_summary)

,Split,Rows,Percent,Start,End
0,Train,35667,79.98,2006-12-31,2022-08-16
1,Validation,4464,10.01,2022-08-17,2024-07-16
2,Test,4464,10.01,2024-07-17,2026-06-06


In [20]:
# Establish Baselines on the basis of 1. Picking the higher ranked player, 2. the bookmaker's odds against actual match victory.

# Ties are resolved simply by choosing Player 1.
# A smaller rank number means a higher-ranked player.

rank_preds = wta["Player_1"].where(
    wta["Rank_1"] <= wta["Rank_2"],
    wta["Player_2"],
)

bookmaker_prob_preds = wta["Player_1"].where(
    wta["Prob_1"] >= wta["Prob_2"],
    wta["Player_2"],
)



In [23]:
rank_accuracy = rank_preds.loc[test.index].eq(test["Winner"]).mean()
bookmaker_accuracy = bookmaker_prob_preds.loc[test.index].eq(test["Winner"]).mean()
rank_accuracy_train = rank_preds.loc[train.index].eq(train["Winner"]).mean()
bookmaker_accuracy_train = bookmaker_prob_preds.loc[train.index].eq(train["Winner"]).mean()

print(f"Rank baseline accuracy (test/train splits): {rank_accuracy:.2%}, {rank_accuracy_train:.2%}")
print(f"Bookmaker baseline accuracy (test/train splits): {bookmaker_accuracy:.2%}, {bookmaker_accuracy_train:.2%}")

Rank baseline accuracy (test/train splits): 63.13%, 65.49%
Bookmaker baseline accuracy (test/train splits): 68.62%, 69.26%


In [ ]:
# Compute additional metrics such as accuracy, precision, recall, and f1 score.

In [ ]:
# Make histograms.